In [71]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix
from keras.datasets import fashion_mnist
import datetime

In [72]:
%load_ext autoreload
%autoreload 2
from Model import NeuralNet, InputLayer, DenseLayer, Sigmoid, Tanh, ReLU, Softmax, OneHotEncoder, MinMaxScaler

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [73]:
import wandb

In [74]:
# Data loading and preprocessing
def load_and_preprocess_data():
    print("Loading Fashion MNIST data...")
    (train_images, train_labels), (test_images, test_labels) = fashion_mnist.load_data()
    
    # Split validation set
    train_images, val_images, train_labels, val_labels = train_test_split(
        train_images, train_labels, test_size=0.2, random_state=42
    )

    # Take small portions of the dataset
    SUBSET_SIZES = {
        'train': 2000,
        'val': 500,
        'test': 250
    }

    # Select subsets
    train_images = train_images[:SUBSET_SIZES['train']]
    train_labels = train_labels[:SUBSET_SIZES['train']]
    
    val_images = val_images[:SUBSET_SIZES['val']]
    val_labels = val_labels[:SUBSET_SIZES['val']]
    
    test_images = test_images[:SUBSET_SIZES['test']]
    test_labels = test_labels[:SUBSET_SIZES['test']]

    # Reshape and scale data
    def process_images(images, scaler=None):
        # Flatten images to (num_samples, 784)
        flattened = images.reshape(images.shape[0], -1)
        
        # Scale using MinMaxScaler
        if scaler is None:
            scaler = MinMaxScaler()
            scaled = scaler.fit_transform(flattened)
            return scaled.T, scaler  # Return transposed data and scaler for validation/test
        else:
            return scaler.transform(flattened).T  # Return transposed data

    # Fit scaler on training data
    train_features, scaler = process_images(train_images)
    
    # Transform validation and test data
    val_features = process_images(val_images, scaler)
    test_features = process_images(test_images, scaler)

    return (
        train_features,
        val_features,
        test_features,
        train_labels,
        val_labels,
        test_labels
    )

In [75]:
# Activation function mapper
def get_activation(activation_name):
    return {
        'Sigmoid': Sigmoid(),
        'Tanh': Tanh(),
        'ReLU': ReLU()
    }[activation_name]

In [76]:
# Training and evaluation
def train_and_evaluate(config=None):
    with wandb.init(config=config):
        config = wandb.config
        
        # Load and prepare data
        X_train, X_val, X_test, y_train, y_val, y_test = load_and_preprocess_data()
        
        # Encode labels
        encoder = OneHotEncoder()
        train_targets = encoder.fit_transform(y_train, 10)
        val_targets = encoder.transform(y_val)
        test_targets = encoder.transform(y_test)
        
        # Create network with multiple hidden layers
        layers = [InputLayer(data=X_train)]
        for i in range(config.num_hidden_layers):
            layers.append(
                DenseLayer(
                    units=config.size_hidden_layer,
                    activation=get_activation(config.activation),
                    name=f"Hidden_{i+1}"
                )
            )
        layers.append(DenseLayer(units=10, activation=Softmax(), name="Output"))
        
        # Initialize model
        model = NeuralNet(
            layers=layers,
            batch_size=config.batch_size,
            optimizer_name=config.optimizer,
            init_method=config.weight_init,
            epochs=config.num_epochs,
            targets=train_targets,
            loss_type=config.loss,
            X_val=X_val,
            targets_val=val_targets,
            use_wandb=True,
            optimizer_params = {
                "learning_rate": config.learning_rate,
                "momentum": 0.9,
                "beta": 0.9,      # for RMSProp
                "beta1": 0.9,     # for Adam/Nadam
                "beta2": 0.999,   # for Adam/Nadam
                "epsilon": 1e-7,
                "weight_decay": config.weight_decay
            }
        )
        
        # Training
        training_history = model.backward_pass()
        
        # Evaluation
        val_acc, val_loss, _ = model.evaluate(X_val, val_targets)
        test_acc, test_loss, _ = model.evaluate(X_test, test_targets)
        
        # Log metrics
        wandb.log({
            "val_loss": val_loss,
            "val_accuracy": val_acc / val_targets.shape[1],
            "test_loss": test_loss,
            "test_accuracy": test_acc / test_targets.shape[1],
            "created": datetime.datetime.now().isoformat()
        })

In [77]:
# Sweep configuration
sweep_config = {
    "name": "complete-sweep",
    "method": "bayes",
    "metric": {"name": "val_loss", "goal": "minimize"},
    "parameters": {
        "num_epochs": {"values": [5, 10]},
        "num_hidden_layers": {"values": [3, 4, 5]},
        "size_hidden_layer": {"values": [32, 64, 128]},
        "weight_decay": {"values": [0, 0.0005, 0.5]},
        "learning_rate": {"values": [1e-3, 1e-4]},
        "optimizer": {"values": ["SGD", "Momentum", "Nesterov", "RMSProp", "Adam", "Nadam"]},
        "batch_size": {"values": [16, 32, 64]},
        "weight_init": {"values": ["Random", "Xavier"]},
        "activation": {"values": ["Sigmoid", "Tanh", "ReLU"]},
        "loss": {"values": ["CrossEntropy"]}
    }
}

In [78]:
# Sweep configuration for Experiment
sweep_config1 = {
    "name": "complete-sweep",
    "method": "grid",
    "metric": {"name": "val_loss", "goal": "minimize"},
    "parameters": {
        "num_epochs": {"values": [10]},
        "num_hidden_layers": {"values": [3]},
        "size_hidden_layer": {"values": [128]},
        "weight_decay": {"values": [0]},
        "learning_rate": {"values": [1e-3]},
        "optimizer": {"values": ["Nesterov"]},
        "batch_size": {"values": [64]},
        "weight_init": {"values": ["Random"]},
        "activation": {"values": ["Sigmoid"]},
        "loss": {"values": ["CrossEntropy"]}
    }
}

In [79]:
def run_experiment():
    sweep_id = wandb.sweep(sweep_config1, project="fashion-mnist-classification")
    wandb.agent(sweep_id, function=train_and_evaluate)

In [80]:
if __name__ == "__main__":
    run_experiment()

Create sweep with ID: 3e6kd1pn
Sweep URL: https://wandb.ai/mrsagarbiswas-iit-madras/fashion-mnist-classification/sweeps/3e6kd1pn


wandb: Agent Starting Run: twrcmp2p with config:
wandb: 	activation: Sigmoid
wandb: 	batch_size: 64
wandb: 	learning_rate: 0.001
wandb: 	loss: CrossEntropy
wandb: 	num_epochs: 10
wandb: 	num_hidden_layers: 3
wandb: 	optimizer: Nesterov
wandb: 	size_hidden_layer: 128
wandb: 	weight_decay: 0
wandb: 	weight_init: Random


Loading Fashion MNIST data...


100%|██████████████████████████████████████████████████████████████████████████████████| 10/10 [00:02<00:00,  4.25it/s]


epoch,▁▂▃▃▄▅▆▆▇█
test_accuracy,▁
test_loss,▁
train_accuracy,▁▁▁▁▁▁▁▁▁▁
train_loss,▁▁▁▁▁▁▁▁▁▁
val_accuracy,▁▁▁▁▁▁▁▁▁▁▁
val_loss,▁▁▁▁▁▁▁▁▁▁█
created,2025-03-10T10:00:40....
epoch,9
test_accuracy,0.072
test_loss,597.28751


wandb: Sweep Agent: Waiting for job.
wandb: Sweep Agent: Exiting.
